# Wind Turbine Anomaly Detection - Anomaly Transformer
## Load Dataset, Run Inference, and Plot Results Over Time

This notebook loads the wind turbine dataset and trained Anomaly Transformer model checkpoints, runs inference on test data, and plots anomaly detection results over time.

## 1. Install and Import Required Libraries

In [ ]:
import sys
import os

# Add the project root and model source directories to the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
model_src = os.path.join(project_root, 'src', 'models', 'wind_turbine', 'anomaly_transformer')
data_loader_src = os.path.join(project_root, 'src')

sys.path.insert(0, project_root)
sys.path.insert(0, model_src)
sys.path.insert(0, data_loader_src)

print("Project root:", project_root)
print("Model source:", model_src)
print("Data loader source:", data_loader_src)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib
from scipy.signal import find_peaks
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# Custom modules
from model.AnomalyTransformer import AnomalyTransformer
from data_loader.turbine_data_loader import get_turbine_data_loader

print("All libraries imported successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: cpu")

## 2. Define Configuration Parameters

Define all required parameters for data loading, model instantiation, and inference. Adjust `model_id` to select the turbine component:
- `'0'`: Generator Bearing
- `'1'`: Gearbox
- `'2'`: Transformer
- `'3'`: Hydraulic

In [ ]:
# ===================== CONFIGURATION =====================
# Change model_id to select the component to analyze
model_id = '0'  # Options: '0' (Generator Bearing), '1' (Gearbox), '2' (Transformer), '3' (Hydraulic)

# Component name mapping for plot titles
component_names = {
    '0': 'Generator Bearing',
    '1': 'Gearbox',
    '2': 'Transformer',
    '3': 'Hydraulic'
}
component_name = component_names[model_id]

# Dataset parameters
dataset_name = 'Wind Farm A'
win_size = 144
batch_size = 64
input_c = 82
output_c = 82
k = 3
anormly_ratio = 1.5
temperature = 50

# Paths - adjust these to your actual directory structure
data_path = os.path.join(project_root, 'datasets', 'test')
model_save_path = os.path.join(project_root, 'model_checkpoints')
scaler_path = os.path.join(project_root, 'scalers')

# Device
device = torch.device('cpu')

# Threshold map matching solver.py for each model class at anormly_ratio=1.5
threshmap = {
    1.5: {
        '0': 0.00013377491304709106,
        '1': 0.00044517662157886675,
        '2': 0.00014006024604896057,
        '3': 7.12080005177995e-05
    }
}

# Get threshold for the selected model
thresh = threshmap[anormly_ratio][model_id]

# Print configuration
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Component:       {component_name} (model_id={model_id})")
print(f"Dataset:         {dataset_name}")
print(f"Window size:     {win_size}")
print(f"Batch size:      {batch_size}")
print(f"Input channels:  {input_c}")
print(f"Output channels: {output_c}")
print(f"k (loss weight): {k}")
print(f"Anomaly ratio:   {anormly_ratio}")
print(f"Threshold:       {thresh:.15f}")
print(f"Temperature:     {temperature}")
print(f"Data path:       {data_path}")
print(f"Model path:      {model_save_path}")
print(f"Scaler path:     {scaler_path}")
print(f"Device:          {device}")
print("=" * 60)

## 3. Load Scaler and Data Loaders

Load the saved StandardScaler and create the test DataLoader using `get_turbine_data_loader`.

In [ ]:
# Load the saved scaler
scaler_file = os.path.join(scaler_path, 'scaler.joblib')
scaler = joblib.load(scaler_file)
print(f"Scaler loaded from: {scaler_file}")
print(f"Scaler mean shape: {scaler.mean_.shape}")
print(f"Scaler scale shape: {scaler.scale_.shape}")

# Construct model identifier (e.g., 'Wind Farm A0')
model_label = dataset_name + model_id

# Load the test data
test_loader, features, _, _ = get_turbine_data_loader(
    data_path=data_path,
    model=model_label,
    batch_size=batch_size,
    win_size=win_size,
    step=win_size,
    mode='test',
    scaler=scaler
)

print(f"\nFeature count: {len(features)}")
print(f"Features: {features[:10]}... (showing first 10)")
print(f"Number of test batches: {len(test_loader)}")

## 4. Build and Load Anomaly Transformer Model from Checkpoint

Instantiate the AnomalyTransformer and load the trained weights from the saved checkpoint.

In [ ]:
# Build the Anomaly Transformer model
model = AnomalyTransformer(
    win_size=win_size,
    enc_in=input_c,
    c_out=output_c,
    e_layers=3
)

# Load checkpoint
checkpoint_file = os.path.join(model_save_path, f"{dataset_name}{model_id}_checkpoint1_.pth")
print(f"Loading checkpoint from: {checkpoint_file}")

state_dict = torch.load(checkpoint_file, map_location=device)
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel loaded successfully!")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Define Loss and Scoring Functions

Define the KL divergence loss function, MSE criterion, and the robust peak detection function used for locating structural peaks in anomaly score sequences.